# KSA NFI Improved V5 - LULC Validation + Trees/Shrubs Calculations (National)

**Built upon:** `F:\1.KSA\1.Forest\10.NFI\10.LULC_Validation` (office-point ESA/ESRI/NDVI validation, forest-reserve rules, Final LULC / New LCU).

**Improved with:** `NFI_ KSA Calculation_Trees&shrubs.docx` (joins, land-use proportions, tree/shrub biomass and carbon chain).

### Scope
Entire Kingdom (national cleansed inventory + national LULC validated points). Not Asir-only.

### Design rules
1. Keep the LULC Validation decision rules and Final LULC as the land-cover authority.
2. Join trees/shrubs to plots/SUs per the calculation document.
3. Apply Word-doc constants only (FF 0.65, WD 0.6, BEF 2, RS 0.28, C 0.47, CO2eq 3.67, CF=10).
4. Sample statistics only; national stock totals BLOCKED without design weights.
5. Field heights authoritative (no remote overwrite).


## 0. Locate project (Drive or auto-clone from GitHub)


In [ ]:
from pathlib import Path
import os, sys, json, warnings, subprocess, shutil
warnings.filterwarnings('ignore')

IN_COLAB = False
try:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print('Drive mount skipped/failed:', e)
    IN_COLAB = True
except Exception:
    pass

REPO_URL = 'https://github.com/gabrielsanya-ops/NFI_LULC_Biomass_Improved.git'
CLONE_DIR = Path('/content/NFI_LULC_Biomass_Improved')
ALIASES = ['NFI_LULC_Biomass_Improved', '11.NFI_LULC_Biomass_Improved']

def looks_like_root(p: Path) -> bool:
    return (p / 'config.yaml').exists() and (p / 'data' / 'lulc').exists() and (p / 'data' / 'cleansed').exists()

CANDIDATES = []
for name in ALIASES:
    CANDIDATES += [
        Path('/content/drive/MyDrive') / name,
        Path('/content/drive/MyDrive/33.Projects') / name,
        Path('/content') / name,
        Path(r'F:/1.KSA/29.NFI') / name,
        Path(r'F:/1.KSA/29.NFI/11.NFI_LULC_Biomass_Improved'),
    ]
CANDIDATES += [Path.cwd(), CLONE_DIR]

ROOT = next((c.resolve() for c in CANDIDATES if c.exists() and looks_like_root(c)), None)
if ROOT is None and IN_COLAB:
    print('Cloning improved package from GitHub...')
    if CLONE_DIR.exists():
        shutil.rmtree(CLONE_DIR, ignore_errors=True)
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_DIR)])
    ROOT = CLONE_DIR.resolve()

if ROOT is None or not looks_like_root(ROOT):
    raise FileNotFoundError('Project root not found. Sync folder to Drive or ensure GitHub repo is public.')

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('IN_COLAB', IN_COLAB, 'ROOT', ROOT)


## 1. Install and imports


In [ ]:
%pip install -q numpy pandas openpyxl pyyaml scikit-learn matplotlib seaborn
import numpy as np, pandas as pd, yaml, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from nfi_improved.biomass import (
    apply_tree_metrics, exclude_by_substrings, shrub_agb_mg, tree_bgb_mg,
    carbon_mg, total_biomass_mg, FORM_FACTOR, WOOD_DENSITY, BEF, ROOT_SHOOT, CARBON_FRACTION,
    TREE_EXCLUDE_SUBSTRINGS, SHRUB_EXCLUDE_SUBSTRINGS, CONVERSION_FACTOR_HA,
)
from nfi_improved.lulc import find_col, lulc_validation_summary, land_use_proportion_by_su, plot_uid

sns.set_theme(style='whitegrid'); RS = 42; np.random.seed(RS)

def resolve(p):
    p = Path(p); return p if p.is_absolute() else (ROOT / p).resolve()

with open(ROOT / 'config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
for section in ('paths', 'tables'):
    for k, v in list(cfg.get(section, {}).items()):
        cfg[section][k] = str(resolve(v))

OUT = Path(cfg['paths']['output_root'])
for sub in ['tables', 'figures', 'maps', 'logs']:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

print('built_upon', cfg['project']['built_upon'])
print('constants FF,WD,BEF,RS,CF =', FORM_FACTOR, WOOD_DENSITY, BEF, ROOT_SHOOT, CARBON_FRACTION)


## 2. Load LULC Validation baseline (national office points)


In [ ]:
lulc = pd.read_csv(cfg['paths']['lulc_validated'])
print('LULC validated rows', len(lulc), 'cols', lulc.shape[1])
summary = lulc_validation_summary(lulc)
print(json.dumps(summary, indent=2))
pd.DataFrame([summary]).to_csv(OUT/'tables'/'lulc_baseline_summary.csv', index=False)

final_col = find_col(lulc, ['final_lulc', 'New LCU'])
display(lulc[final_col].value_counts().head(20).rename('n').to_frame() if final_col else None)


## 3. Ingest cleansed national tables (Word workflow joins)


In [ ]:
tables = {k: pd.read_excel(v) for k, v in cfg['tables'].items()}
for k, v in tables.items():
    print(f'{k:12s} n={len(v):6d} cols={v.shape[1]}')

su = tables['su']; plot = tables['plot']; trees = tables['trees']; shrubs = tables['shrubs']
su_id = find_col(su, [cfg['columns']['su_id_in_su_master'], 'su_no'])
region_col = find_col(su, [cfg['columns']['region_en'], 'su_group_region_e'])
child_su = find_col(plot, [cfg['columns']['su_id_in_child'], 'nfi_of_ksa_program_su_su_no', 'su_no'])
tree_su = find_col(trees, [cfg['columns']['su_id_in_child'], 'nfi_of_ksa_program_su_su_no', 'su_no'])
tree_plot = find_col(trees, ['plot_plot_id1', 'plot_id1', 'plot_id'])
dbh_col = find_col(trees, [cfg['columns']['dbh'], 'tree_dbh'])
ht_col = find_col(trees, [cfg['columns']['height'], 'tree_height'])
species_col = find_col(trees, ['tree_species_scientific_name', 'tree_species', 'species', 'scientific'])
plot_no_col = find_col(plot, [cfg['columns']['plot_id'], 'plot_id1', 'plot_no'])

# Align LULC keys
lulc_su = find_col(lulc, ['su_no'])
lulc_plot = find_col(lulc, ['plot_no'])
lulc['plot_uid'] = plot_uid(lulc[lulc_su], lulc[lulc_plot])

print('regions', su[region_col].nunique() if region_col else None)
if region_col:
    display(su[region_col].value_counts().rename('n_SU').to_frame())


## 4. Land-use proportions by SU (calculation document section 5)

`Land Use Proportion = n plots of class / 5`. Uses **Final LULC** from the LULC Validation baseline.


In [ ]:
# Prefer one Final LULC per plot (first LUCS segment)
plot_lulc = (lulc.sort_values(['su_no', 'plot_no'])
             .drop_duplicates(['su_no', 'plot_no'], keep='first')
             [['su_no', 'plot_no', final_col, 'lon', 'lat', 'validation_true', 'inside_forest_reserved']]
             .rename(columns={final_col: 'final_lulc'}))
print('unique office plots with Final LULC', len(plot_lulc))

lu_prop = land_use_proportion_by_su(plot_lulc, 'su_no', 'final_lulc', plots_per_su=cfg['project']['plots_per_su'])
display(lu_prop.head(20))
lu_prop.to_csv(OUT/'tables'/'national_land_use_proportion_by_su.csv', index=False)

# National class share (sample of plots, not design-based area)
class_share = plot_lulc['final_lulc'].value_counts(normalize=True).rename('plot_share').to_frame()
class_share['n_plots'] = plot_lulc['final_lulc'].value_counts()
display(class_share.head(15))
class_share.to_csv(OUT/'tables'/'national_final_lulc_plot_share.csv')


## 5. Trees: QC exclusions + provisional biomass (Word section 6)


In [ ]:
if species_col:
    keep = exclude_by_substrings(trees[species_col], TREE_EXCLUDE_SUBSTRINGS)
    trees_live = trees.loc[keep].copy()
    n_excl = int((~keep).sum())
else:
    trees_live = trees.copy(); n_excl = 0

valid = trees_live[dbh_col].notna() & trees_live[ht_col].notna()
trees_m = apply_tree_metrics(trees_live.loc[valid].copy(), dbh_col, ht_col)
trees_m['su_key'] = trees_m[tree_su].astype(str).str.strip()
if tree_plot is not None:
    trees_m['plot_uid'] = plot_uid(trees_m[tree_su], trees_m[tree_plot])

# Attach region + Final LULC where plot_uid matches
su_map = su[[su_id] + ([region_col] if region_col else [])].copy()
su_map[su_id] = su_map[su_id].astype(str).str.strip()
rename_map = {su_id: 'su_key'}
if region_col:
    rename_map[region_col] = 'region'
su_map = su_map.rename(columns=rename_map)
trees_m = trees_m.merge(su_map, on='su_key', how='left')
if 'plot_uid' in trees_m.columns:
    trees_m = trees_m.merge(plot_lulc[['su_no', 'plot_no', 'final_lulc']].assign(
        plot_uid=lambda d: plot_uid(d['su_no'], d['plot_no'])
    )[['plot_uid', 'final_lulc']], on='plot_uid', how='left')

print(f'excluded_species={n_excl} trees_with_metrics={len(trees_m)}')
display(trees_m[[dbh_col, ht_col, 'basal_area_m2', 'volume_m3', 'agb_mg', 'carbon_mg_provisional']].describe().T)
trees_m.to_csv(OUT/'tables'/'national_trees_with_provisional_biomass.csv', index=False)


## 6. SU tree density (anti-clip; Word density formulas)


In [ ]:
national_su_ids = su[su_id].astype(str).str.strip().tolist()
frame = pd.DataFrame({'su_key': national_su_ids}).drop_duplicates()
counts = trees_m.groupby('su_key').size().rename('n_tree_records').reset_index()
su_tree = frame.merge(counts, on='su_key', how='left')
su_tree['n_tree_records'] = su_tree['n_tree_records'].fillna(0).astype(int)
su_tree['has_trees'] = su_tree['n_tree_records'] > 0
# average tree density across 5 plots, then * CF=10 for trees/ha (document)
su_tree['avg_trees_per_plot'] = su_tree['n_tree_records'] / float(cfg['project']['plots_per_su'])
su_tree['tree_density_ha'] = su_tree['avg_trees_per_plot'] * CONVERSION_FACTOR_HA
su_tree = su_tree.merge(su_map, on='su_key', how='left')

g1 = {
    'scope': 'national_KSA',
    'n_su': int(len(su_tree)),
    'n_su_with_trees': int(su_tree['has_trees'].sum()),
    'n_su_zero_trees_retained': int((~su_tree['has_trees']).sum()),
    'n_trees_metrics': int(len(trees_m)),
    'note': 'ANTI_CLIP_SAMPLE_NOT_DESIGN_BASED',
}
print(json.dumps(g1, indent=2))
if 'region' in su_tree.columns:
    display(su_tree.groupby('region').agg(n_su=('su_key','size'), with_trees=('has_trees','sum')).sort_values('n_su', ascending=False))
su_tree.to_csv(OUT/'tables'/'national_su_tree_density_anticlip.csv', index=False)
pd.DataFrame([g1]).to_csv(OUT/'tables'/'national_gate_tree_presence.csv', index=False)


## 7. Shrubs provisional biomass (Word section 8)


In [ ]:
sh = shrubs.copy()
sh_su = find_col(sh, [cfg['columns']['su_id_in_child'], 'nfi_of_ksa_program_su_su_no', 'su_no'])
sp = find_col(sh, ['species', 'shrub_species', 'sp_name'])
cd1 = find_col(sh, ['crown_diam1', 'crown_diameterl1', 'crown_diameter_1', 'cd1'])
cd2 = find_col(sh, ['crown_diam2', 'crown_diameterl2', 'crown_diameter_2', 'cd2'])
sh_h = find_col(sh, ['sh_height', 'shrub_height', 'height', 'height_m', 'tree_height'])
shrub_note = 'shrubs not processed'
if sp is not None:
    sh = sh.loc[exclude_by_substrings(sh[sp], SHRUB_EXCLUDE_SUBSTRINGS)]
if cd1 and cd2 and sh_h and len(sh):
    sh = sh.copy()
    sh['mean_CD_m'] = (pd.to_numeric(sh[cd1], errors='coerce') + pd.to_numeric(sh[cd2], errors='coerce')) / 2.0
    h_m = pd.to_numeric(sh[sh_h], errors='coerce')
    sh['height_cm'] = np.where(h_m.median(skipna=True) < 20, h_m * 100.0, h_m)
    ok = sh['mean_CD_m'].notna() & sh['height_cm'].notna() & (sh['mean_CD_m'] > 0) & (sh['height_cm'] > 0)
    sh_ok = sh.loc[ok].copy()
    sh_ok['agb_mg'] = shrub_agb_mg(sh_ok['mean_CD_m'], sh_ok['height_cm'])
    sh_ok['bgb_mg'] = tree_bgb_mg(sh_ok['agb_mg'])
    sh_ok['total_biomass_mg'] = total_biomass_mg(sh_ok['agb_mg'], sh_ok['bgb_mg'])
    sh_ok['carbon_mg_provisional'] = carbon_mg(sh_ok['total_biomass_mg'])
    sh_ok.to_csv(OUT/'tables'/'national_shrubs_with_provisional_biomass.csv', index=False)
    shrub_note = f'shrubs processed n={len(sh_ok)}'
print(shrub_note)


## 8. Height~DBH model (SU-blocked CV; no circular BA feature)


In [ ]:
X = trees_m[[dbh_col]].to_numpy()
y = trees_m[ht_col].to_numpy()
groups = trees_m['su_key'].astype(str).to_numpy()
ols = LinearRegression().fit(X, y)
print('OLS R2', float(r2_score(y, ols.predict(X))), 'RMSE', float(np.sqrt(mean_squared_error(y, ols.predict(X)))))

gkf = GroupKFold(n_splits=5)
rows = []
for fold, (tr, te) in enumerate(gkf.split(X, y, groups), 1):
    rf = RandomForestRegressor(n_estimators=200, random_state=RS, n_jobs=-1, min_samples_leaf=5)
    rf.fit(X[tr], y[tr])
    pred = rf.predict(X[te])
    rows.append({'fold': fold, 'r2': float(r2_score(y[te], pred)),
                 'rmse': float(np.sqrt(mean_squared_error(y[te], pred))),
                 'mae': float(mean_absolute_error(y[te], pred))})
cv = pd.DataFrame(rows)
display(cv)
print('GroupKFold mean RMSE', float(cv.rmse.mean()))
cv.to_csv(OUT/'tables'/'national_rf_groupkfold_height_dbh.csv', index=False)


## 9. Figures and national map


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), dpi=120)
ax.bar(['LULC accepted', 'LULC updated'], [summary['accepted_true'], summary['updated_false']],
       color=['#2e7d32', '#ef6c00'])
ax.set_title('Fig 1. LULC Validation baseline (national office points)')
ax.set_ylabel('Plots / rows')
fig.tight_layout(); fig.savefig(OUT/'figures'/'fig01_lulc_baseline.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(7, 5), dpi=120)
ax.hist(trees_m[dbh_col].dropna(), bins=40, color='#2f4f4f', edgecolor='white')
ax.set_xlabel('DBH (cm)'); ax.set_ylabel('Count')
ax.set_title('Fig 2. National tree DBH (sample; calc-doc metrics)')
fig.tight_layout(); fig.savefig(OUT/'figures'/'fig02_national_dbh.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
ax.bar(['SUs with trees', 'SUs zero trees retained'],
       [g1['n_su_with_trees'], g1['n_su_zero_trees_retained']], color=['#2e7d32', '#9e9e9e'])
ax.set_title('Fig 3. National anti-clip SU tree presence'); ax.set_ylabel('Count')
fig.tight_layout(); fig.savefig(OUT/'figures'/'fig03_national_anticlip.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(9, 8), dpi=120)
ax.scatter(plot_lulc['lon'], plot_lulc['lat'], s=3, c='#0b3d0b', alpha=0.35)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Map 1. National NFI office plots (LULC Validation coordinates)\\nCRS: WGS84')
ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(OUT/'maps'/'map01_national_office_plots.png', dpi=200); plt.show()
plot_lulc.to_csv(OUT/'tables'/'national_office_plots_final_lulc.csv', index=False)


## 10. Export run summary


In [ ]:
run = {
    'project': cfg['project']['name'],
    'version': cfg['project']['version'],
    'built_upon': cfg['project']['built_upon'],
    'scope': 'national_KSA',
    'lulc_summary': summary,
    'tree_presence': g1,
    'n_trees_metrics': int(len(trees_m)),
    'shrub_note': shrub_note,
    'policy': cfg['policy'],
    'ROOT': str(ROOT),
    'assumptions': [
        'Shrub allometric log base assumed natural (ln); source unclear.',
        'Total biomass = AGB + BGB implied (not explicit equation in Word doc).',
        'National stock totals blocked without design weights / SU area.',
        'LULC Final labels from 10.LULC_Validation remain the land-cover authority.',
    ],
}
(OUT/'logs'/'run_summary.json').write_text(json.dumps(run, indent=2, default=str), encoding='utf-8')
print(json.dumps(run, indent=2, default=str))
print('V5 LULC + biomass improved pipeline complete.')
